In [2]:
import os
import hashlib
import logging
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
import boto3
from botocore.exceptions import ClientError
import requests
from io import BytesIO
import warnings

warnings.filterwarnings('ignore')

logger = logging.getLogger(__name__)

In [3]:
import pandas as pd
import pandera.pandas as pa
from pandera import Check, Column, DataFrameSchema
import re

realty_schema = DataFrameSchema(
    columns={
        "offer_id": Column(str, nullable=False, unique=True, 
                          checks=Check.str_matches(r'^\d+$')),
        
        "price_numeric": Column(float, nullable=True, 
                               checks=Check.in_range(1e5, 1e15), 
                               coerce=True),
        "area": Column(float, nullable=True, coerce=True, 
                      checks=Check.gt(0)),

        "rooms": Column(str, nullable=True,
                       checks=Check.str_matches(r'^(\d+|студия|N/A|multiroom)$', ignore_na=True)),

        "self_floor": Column(float, nullable=True,
                       checks=Check.in_range(-5, 100),
                       coerce=True),
                       
        "max_floor": Column(float, nullable=True,
                       checks=Check.in_range(-5, 100),
                       coerce=True),
        
        "metro_time": Column(float, nullable=True,
                            checks=Check.in_range(0, 1e4),
                            coerce=True),
        
        "url": Column(str, nullable=False,
                     checks=Check.str_matches(r'^https?://.+', ignore_na=True)),

        "photo_count": Column(float, nullable=True, 
                             checks=Check.between(0, 100),
                             coerce=True),
    },
    strict=False,
    coerce=True,  
    unique=["offer_id"],
)


In [4]:
import pandas as pd
import pandera.pandas as pa
from pandera.errors import SchemaError
import logging
import re

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def metro_time_cleaner(text):
    if str(text) == 'nan':
        return None
    try:
        num, time_type = text.split()
    except Exception:
        print(text)
    if time_type=='час':
        return int(num)*60
    return int(num)

def floor_tipization(text, self_number=True):
    if str(text) == 'nan':
        return None
    try:
        self_floor, max_floor = text.split(' этаж из ')
    except Exception:
        self_floor, max_floor = text.split(' этаж')[0], text.split(' этаж')[0]
    if self_number:
        if ',' in self_floor:
            return int(self_floor.split(',')[0])
        
    return int(max_floor)

def parse_price_rub(price_str: str) -> float | None:
    if pd.isna(price_str) or price_str == 'N/A':
        return None
    digits = re.sub(r'[^\d]', '', str(price_str))
    return float(digits) if digits else None

def rooms_cleaner(text):
    if str(text) == 'nan':
        return None
    if text=='студия':
        return 0
    return int(text)

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()
    
    mask = df_clean['price_numeric'].isna() & df_clean['price'].notna()
    df_clean.loc[mask, 'price_numeric'] = df_clean.loc[mask, 'price'].apply(parse_price_rub)
    
    df_clean['metro_time'] = df_clean['metro_time'].apply(metro_time_cleaner)
    df_clean['self_floor'] = df_clean['floor'].apply(floor_tipization, self_number=True)
    df_clean['max_floor'] = df_clean['floor'].apply(floor_tipization, self_number=False)

    df_clean['price_per_m2'] = df_clean['price_per_m2'].apply(lambda x: int(x.split()[0]) if str(x)!='nan' else None)
    df_clean = df_clean.drop_duplicates(subset=['offer_id'], keep='first')
    
    df_clean = df_clean.dropna(subset=['offer_id', 'url'])
    
    return df_clean

def validate_and_filter(df: pd.DataFrame, schema: pa.DataFrameSchema):
    df_valid = schema.validate(df, lazy=False)
    return df_valid, pd.DataFrame()

def main(input_path, output_path):
    df_raw = pd.read_csv(input_path)
    
    df_cleaned = clean_dataframe(df_raw)
    logger.info(f"After cleaning: {len(df_cleaned)} records")
    df_valid, df_invalid = validate_and_filter(df_cleaned, realty_schema)
    
    if not df_invalid.empty:
        logger.warning(f"✗ Invalid records: {len(df_invalid)}")
        df_invalid.to_csv(output_path.replace('.parquet', '_invalid.csv'), index=False)
    
    logger.info(f"Saving to {output_path}")
    df_valid.to_parquet(
        output_path,
        index=False
    )
    
    stats = {
        'total_raw': len(df_raw),
        'after_cleaning': len(df_cleaned),
        'valid_final': len(df_valid),
        'invalid_dropped': len(df_invalid),
        'columns': list(df_valid.columns)
    }
    logger.info(f"Pipeline stats: {stats}")
    
    return df_valid

if __name__ == "__main__":
    input_path = '/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/yandex_realty_manual.csv'
    output_path = '/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/yandex_realty_cleaned.parquet'
    
    main(input_path, output_path)

INFO:__main__:After cleaning: 93667 records
INFO:__main__:Saving to /Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/yandex_realty_cleaned.parquet
INFO:__main__:Pipeline stats: {'total_raw': 93667, 'after_cleaning': 93667, 'valid_final': 93667, 'invalid_dropped': 0, 'columns': ['offer_id', 'price', 'price_numeric', 'old_price', 'area', 'rooms', 'floor', 'price_per_m2', 'metro', 'metro_time', 'address', 'author', 'main_image', 'photo_count', 'badges', 'publish_date', 'url', 'title', 'description', 'image_urls', 'self_floor', 'max_floor']}


In [ ]:
import pandas as pd
from s3_client import MinIOS3Client

client = MinIOS3Client(
    endpoint_url='http://localhost:9000',
    access_key='minioadmin',
    secret_key='minioadmin123'
)

client.create_bucket('tabular-data')
client.create_bucket('embeddings')
client.create_bucket('photos')

In [ ]:
df_cleaned = pd.read_parquet('/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/yandex_realty_cleaned.parquet')
print(f"Cleaned data shape: {df_cleaned.shape}")

client.upload_dataframe(
    df=df_cleaned,
    bucket_name='tabular-data',
    object_key='yandex_realty_cleaned.parquet',
    format='parquet'
)

In [ ]:
df_embeddings = pd.read_parquet('/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/ya_realty_with_txt_embeds.parquet')
print(f"Embeddings data shape: {df_embeddings.shape}")

client.upload_dataframe(
    df=df_embeddings,
    bucket_name='embeddings',
    object_key='ya_realty_with_txt_embeds.parquet',
    format='parquet'
)


In [ ]:
# df_downloaded = client.download_dataframe(
#     bucket_name='tabular-data',
#     object_key='yandex_realty_cleaned.parquet',
#     format='parquet'
# )
# print(f"\n Downloaded data shape: {df_downloaded.shape}")

# client.download_image(
#     bucket_name='photos',
#     object_key='realty_photos/photo_001.jpg',
#     save_path='/path/to/downloaded_photo.jpg'
# )

# image = client.get_image_as_pil(
#     bucket_name='photos',
#     object_key='realty_photos/photo_001.jpg'
# )
# if image:
#     print(f"Image size: {image.size}")

# exists = client.object_exists(
#     bucket_name='tabular-data',
#     object_key='yandex_realty_cleaned.parquet'
# )
# print(f"\n Object exists: {exists}")

# all_buckets = client.list_buckets()